<a href="https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/fix/madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QuickStart CityLearn v3 MADRL

Este notebook sigue la estructura del `quickstart.ipynb` original de CityLearn, adaptado al proyecto **CityLearn v3 MADRL**. El simulador base sigue siendo CityLearn v2, y la capa v3 agrega Dec-POMDP, CTDE, 17 edificios + EV, tres ejes `E1/E2/E3`, recompensa `CityLearnV3MADRLRewardFunction`, KPIs por eje y backends HAPPO, MASAC, MATD3 y MAAC.

Los ejemplos usan el dataset congelado de la campaña nueva, split `train_2023` de `data/campaigns/iquitos_v5/`. El entrenamiento se lanza solo con la CLI `uc3m.application.training.cli` y la configuracion `configs/experiments/iquitos.yaml`; este QuickStart nunca entrena por ejecutar todas sus celdas.

En local se recomienda usar el entorno `.venv39-citylearn-v3`. En Google Colab se clona el repositorio padre con sus submodulos fijados. La instalacion queda apagada por defecto.

In [ ]:
# Controla si se clona e instala el proyecto en Google Colab.
RUN_COLAB_INSTALL = False

# Importa utilidades del sistema operativo.
import os

# Importa rutas portables para Windows, Linux y Colab.
from pathlib import Path

# Ejecuta comandos externos cuando una celda lo requiere.
import subprocess

# Expone el interprete Python activo.
import sys

# Clona el repositorio padre con sus submodulos fijados solo si se activa la bandera.
if RUN_COLAB_INSTALL:
    # Clona el proyecto completo: CityLearn y external/ quedan en los commits fijados por el padre.
    subprocess.run(['git', 'clone', '--recursive', 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'], check=True)
    # Cambia el directorio activo a la raiz del proyecto clonado.
    os.chdir('MADRLCitytleranflexresdr')
    # Instala CityLearn en modo editable (la instalacion completa Python 3.9 esta en la celda 1.3 del tutorial).
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'CityLearn/'], check=True)
# Mantiene la celda segura cuando no se instala nada.
else:
    # Informa que no se modifico el entorno.
    print('RUN_COLAB_INSTALL=False; usando el entorno actual.')

# Importa display cuando se ejecuta dentro de Jupyter.
try:
    # Usa el display nativo de IPython.
    from IPython.display import display
# Define un fallback para validaciones ejecutadas como script.
except Exception:
    # Usa print cuando IPython no esta disponible.
    display = print

## Project Setup

La siguiente celda detecta la raiz del proyecto padre aunque el notebook se ejecute desde `examples_madrl_v3/` y fija el `schema.json` del split de entrenamiento `train_2023` de la campaña nueva.

In [ ]:
# Importa JSON para leer la salida de la CLI de campaña.
import json

# Detecta la raiz del proyecto padre: contiene uc3m/ y configs/experiments/.
def find_project_root(start=None):
    # Parte del directorio actual del notebook.
    start = Path(start or Path.cwd()).resolve()
    # Recorre el directorio actual, sus padres y la ruta estandar de Colab.
    for candidate in (start, *start.parents, Path('/content/MADRLCitytleranflexresdr')):
        # Acepta la primera carpeta con la estructura del proyecto.
        if (candidate / 'uc3m').is_dir() and (candidate / 'configs' / 'experiments').is_dir():
            return candidate
    # Falla de forma visible si el notebook no esta dentro del proyecto.
    raise FileNotFoundError('No se encontro la raiz del proyecto (uc3m/ + configs/experiments/).')

# Fija la raiz del proyecto padre.
PROJECT_ROOT = find_project_root()

# Fija la raiz del simulador CityLearn (dependencia protegida, solo lectura).
CITYLEARN_ROOT = PROJECT_ROOT / 'CityLearn'

# Agrega proyecto y CityLearn al path de Python del kernel.
for _path in (str(CITYLEARN_ROOT), str(PROJECT_ROOT)):
    if _path not in sys.path:
        sys.path.insert(0, _path)

# Usa el Python 3.9 del proyecto si una celda previa lo definio.
PYTHON = globals().get('PROJECT_PYTHON', sys.executable)

# Configuracion unica de la campaña nueva.
CAMPAIGN_CONFIG = 'configs/experiments/iquitos.yaml'

# Dataset congelado de la campaña (splits train_2023 / validation_2024 / test_2025).
DATASET_ROOT = PROJECT_ROOT / 'data/campaigns/iquitos_v5'

# Raiz de artefactos de la campaña nueva; no existe hasta ejecutar --stage prepare.
ARTIFACT_ROOT = PROJECT_ROOT / 'outputs/campaigns/iquitos_v5'

# Manifiesto de la campaña escrito por la etapa prepare.
MANIFEST_PATH = ARTIFACT_ROOT / 'campaign_manifest.json'

# Entorno de los subprocesos: la CLI importa uc3m y CityLearn desde el arbol local.
def cli_env():
    env = dict(os.environ)
    env['PYTHONPATH'] = os.pathsep.join([str(PROJECT_ROOT), str(CITYLEARN_ROOT), env.get('PYTHONPATH', '')])
    return env

# Muestra el codigo de salida y el JSON que imprime la CLI de campaña.
def show_cli_result(result):
    print('exit code =', result.returncode)
    lines = [line for line in (result.stdout or '').splitlines() if line.strip()]
    if lines:
        try:
            print(json.dumps(json.loads(lines[-1]), indent=2))
        except ValueError:
            print('\n'.join(lines[-20:]))
    if result.returncode != 0 and result.stderr:
        print(result.stderr[-3000:])

# Muestra las rutas clave para diagnostico.
print('PROJECT_ROOT  =', PROJECT_ROOT)
print('CAMPAIGN      =', PROJECT_ROOT / CAMPAIGN_CONFIG)
print('DATASET_ROOT  =', DATASET_ROOT, '(existe)' if DATASET_ROOT.is_dir() else '(FALTA)')
print('ARTIFACT_ROOT =', ARTIFACT_ROOT, '(preparada)' if MANIFEST_PATH.is_file() else '(aun no preparada)')

# Schema del split de entrenamiento congelado (2023, 8760 horas).
schema_path = DATASET_ROOT / 'splits' / 'train_2023' / 'schema.json'

# Falla temprano si el dataset de la campaña no esta disponible.
if not schema_path.is_file():
    raise FileNotFoundError(f'No se encontro el schema de la campaña: {schema_path}')

# Define el horizonte corto del quickstart.
quick_steps = 24

# Define la semilla del ejemplo (no es una semilla de la campaña).
seed = 0

# Muestra el schema activo.
print('schema_path =', schema_path)

## CityLearn Control Agents

El QuickStart original muestra agentes CityLearn internos como baseline, RBC, SAC y MARLISA. En este proyecto esos agentes se conservan como referencia v2, pero la contribucion principal es entrenar controladores MADRL v3 con Dec-POMDP y CTDE.

### No Control Baseline

Esta celda ejecuta el baseline CityLearn v2 sobre el split `train_2023` de la campaña. Se usa `episode_time_steps=24` para que el QuickStart sea rapido; la confirmacion usa 50 episodios de 8760 pasos.

In [ ]:
# Importa el agente baseline de CityLearn v2.
from citylearn.agents.base import BaselineAgent as BaselineAgent

# Importa el entorno base CityLearn v2.
from citylearn.citylearn import CityLearnEnv

# Crea el entorno centralizado de baseline.
baseline_env = CityLearnEnv(str(schema_path), central_agent=True, episode_time_steps=quick_steps, random_seed=seed, offline=True)

# Crea el agente baseline sin control activo.
baseline_model = BaselineAgent(baseline_env)

# Reinicia el entorno.
observations, _ = baseline_env.reset(seed=seed)

# Ejecuta un episodio corto.
while not baseline_env.terminated:
    # Predice acciones baseline.
    actions = baseline_model.predict(observations)
    # Aplica acciones al entorno.
    observations, reward, terminated, truncated, info = baseline_env.step(actions)

# Evalua KPIs CityLearn v2.
baseline_kpis = baseline_env.evaluate()

# Convierte KPIs a una tabla compacta.
baseline_table = baseline_kpis.pivot(index='cost_function', columns='name', values='value').round(3).dropna(how='all')

# Muestra la tabla baseline.
display(baseline_table)

### Centralized RBC

RBC se usa como linea base explicable. Se usa el controlador de referencia con soporte EV y `washing_machine_1`, porque `BasicRBC` no cubre todos los actuadores del dataset de la campaña.

In [ ]:
# Importa RBC de referencia con soporte EV y cargas flexibles.
from citylearn.agents.rbc import BasicElectricVehicleRBC_ReferenceController as ProjectRBC

# Crea el entorno centralizado para RBC.
rbc_env = CityLearnEnv(str(schema_path), central_agent=True, episode_time_steps=quick_steps, random_seed=seed, offline=True)

# Crea el controlador RBC compatible con el dataset actual.
rbc_model = ProjectRBC(rbc_env)

# Reinicia el entorno.
observations, _ = rbc_env.reset(seed=seed)

# Ejecuta un episodio corto.
while not rbc_env.terminated:
    # Predice acciones RBC.
    actions = rbc_model.predict(observations)
    # Aplica acciones al entorno.
    observations, reward, terminated, truncated, info = rbc_env.step(actions)

# Evalua KPIs CityLearn v2.
rbc_kpis = rbc_env.evaluate()

# Convierte KPIs a tabla compacta.
rbc_table = rbc_kpis.pivot(index='cost_function', columns='name', values='value').round(3).dropna(how='all')

# Muestra la tabla RBC.
display(rbc_table)

### CityLearn v3 Dec-POMDP Smoke Test

Esta celda crea el entorno v3 del proyecto. La diferencia clave frente a CityLearn v2 es que ahora cada edificio es un agente, el estado global se expone para CTDE y la recompensa usa pesos por eje y perfil MADRL.

In [ ]:
# Importa utilidades del entorno v3.
from citylearn.v3 import describe_environment, evaluate_objectives, make_citylearn_v3_env

# Crea el entorno Dec-POMDP para el eje E1 con perfil HAPPO sobre el split train_2023.
madrl_env = make_citylearn_v3_env(schema_path=schema_path, scenario='E1', seed=seed, episode_time_steps=quick_steps, madrl_algorithm='HAPPO')

# Describe el entorno v3 creado.
madrl_description = describe_environment(madrl_env)

# Imprime numero de agentes.
print('num_agents:', madrl_description['num_agents'])

# Imprime dimension del estado global CTDE.
print('state_dim:', madrl_description['state_dim'])

# Imprime si hay acciones EV.
print('has_ev_actions:', madrl_description['has_ev_actions'])

# Imprime metadatos de reward v3.
print('reward_metadata:', madrl_description['reward_metadata'])

# Reinicia el entorno multiagente.
observations, infos = madrl_env.reset(seed=seed)

# Construye acciones cero validas por agente.
zero_actions = {agent: madrl_env.action_space(agent).sample() * 0.0 for agent in madrl_env.agents}

# Ejecuta un paso Dec-POMDP.
next_observations, rewards, terminations, truncations, step_infos = madrl_env.step(zero_actions)

# Imprime recompensa media del primer paso.
print('reward_mean_step_1:', sum(rewards.values()) / max(len(rewards), 1))

# Imprime forma del estado CTDE.
print('state_shape:', madrl_env.state().shape)

### Decentralized-Cooperative MADRL: HAPPO

HAPPO usa entrenamiento on-policy y critic centralizado desde HARL, con ejecucion descentralizada por edificio. Los cuatro MADRL se entrenan solo mediante la CLI de campaña: la celda prepara la campaña (`--stage prepare`, sin entrenar) y muestra el tratamiento piloto que la CLI le asigna a HAPPO (E1, semilla 901, 2 episodios de 168 pasos).

In [ ]:
# Prepara la campaña: escribe manifiestos y presupuesto. NO entrena ni consulta la GPU.
prepare_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                   '--config', 'configs/experiments/iquitos.yaml', '--stage', 'prepare']

# Muestra el comando exacto.
print(' '.join(map(str, prepare_command)))

# Ejecuta la preparacion; si la campaña ya existe, la CLI solo verifica que sus entradas no cambiaron.
prepare_result = subprocess.run(prepare_command, cwd=PROJECT_ROOT, env=cli_env(),
                                capture_output=True, text=True, check=False)

# Muestra el JSON de la CLI (status prepared_not_executed y presupuesto de pasos).
show_cli_result(prepare_result)

# Muestra los tratamientos piloto que la CLI preparo para un algoritmo (lectura de stages/pilot.json).
def show_pilot_treatments(algorithm):
    pilot_path = ARTIFACT_ROOT / 'stages' / 'pilot.json'
    if not pilot_path.is_file():
        print('Campaña no preparada: no hay plan piloto que mostrar.')
        return
    rows = [row for row in json.loads(pilot_path.read_text(encoding='utf-8'))['treatments']
            if row.get('algorithm') == algorithm]
    for row in rows:
        print(row.get('treatment_id'), '| escenario', row.get('scenario'), '| semilla', row.get('seed'),
              '| episodios', row.get('episodes'), 'x', row.get('episode_time_steps'), 'pasos')
        print('  argumentos nativos:', ' '.join(map(str, row.get('native_arguments', []))))
    if not rows:
        print(f'{algorithm}: sin tratamientos en el piloto.')

# Muestra el plan piloto de HAPPO.
show_pilot_treatments('HAPPO')

### Decentralized-Cooperative MADRL: MASAC

MASAC usa una variante soft actor-critic multiagente con estado global CTDE y politica local discretizada para acoplarse a CityLearn. La celda muestra su tratamiento piloto preparado por la CLI.

In [ ]:
# Muestra el plan piloto de MASAC preparado por la CLI de campaña.
show_pilot_treatments('MASAC')

### Decentralized-Cooperative MADRL: MATD3

MATD3 usa actores continuos y criticos centralizados con observaciones/acciones conjuntas. Es el backend mas natural para control continuo de almacenamiento, EV y rampas. La celda muestra su tratamiento piloto preparado por la CLI.

In [ ]:
# Muestra el plan piloto de MATD3 preparado por la CLI de campaña.
show_pilot_treatments('MATD3')

### Decentralized-Cooperative MADRL: MAAC

MAAC usa critic de atencion para coordinar agentes. En este proyecto se adapta a la comunidad de edificios con acciones discretizadas y observaciones locales por edificio. La celda muestra su tratamiento piloto preparado por la CLI.

In [ ]:
# Muestra el plan piloto de MAAC preparado por la CLI de campaña.
show_pilot_treatments('MAAC')

## Other Standard Reinforcement Learning Libraries

El QuickStart original incluye Stable-Baselines3 y RLlib. Para CityLearn v3 MADRL, los backends del proyecto viven en `external/` y la CLI de campaña los alcanza mediante el adaptador `uc3m.adapters.algorithms.native_entrypoint`, nunca llamandolos directamente. Aun asi, el proyecto incluye adaptadores para flujos RLlib/MARLlib cuando se requieran experimentos adicionales.

### Stable Baselines3 Reinforcement Learning Algorithms

Stable-Baselines3 no soporta multiagente Dec-POMDP directamente. Se puede usar como baseline centralizado CityLearn v2, pero no representa la contribucion MADRL v3. Por eso esta celda solo imprime el comando de instalacion opcional.

In [ ]:
# Controla si se instala Stable-Baselines3.
INSTALL_SB3 = False

# Construye el comando de instalacion compatible con CityLearn v2.
sb3_install_command = [sys.executable, '-m', 'pip', 'install', 'stable-baselines3<=2.2.1']

# Muestra el comando opcional.
print(' '.join(sb3_install_command))

# Instala SB3 solo si se activa la bandera.
if INSTALL_SB3:
    # Ejecuta la instalacion opcional.
    subprocess.run(sb3_install_command, check=True)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se instalo SB3.
    print('INSTALL_SB3=False; SB3 no es necesario para los cuatro MADRL oficiales.')

### RLlib / MARLlib Adapter

`CityLearnV3MARLlibEnv` expone el Dec-POMDP como `MultiAgentEnv`: rellena observaciones/acciones heterogeneas a dimensiones comunes y mantiene el mapeo por edificio. Esta celda prueba el adaptador sin entrenar RLlib.

In [ ]:
# Importa la configuracion v3 del proyecto.
from citylearn.v3 import CityLearnV3ExperimentConfig, CityLearnV3MARLlibEnv

# Crea una configuracion corta para el adaptador sobre el split train_2023.
marllib_config = CityLearnV3ExperimentConfig(schema_path=schema_path, episode_time_steps=quick_steps)

# Crea el adaptador RLlib/MARLlib para E1.
marllib_env = CityLearnV3MARLlibEnv({'config': marllib_config, 'scenario': 'E1', 'seed': seed})

# Obtiene informacion multiagente del entorno.
env_info = marllib_env.get_env_info()

# Imprime numero de agentes.
print('num_agents:', env_info['num_agents'])

# Imprime limite de episodio.
print('episode_limit:', env_info['episode_limit'])

# Imprime dimensiones originales de los primeros agentes.
print('original_observation_dims_sample:', dict(list(env_info['original_observation_dims'].items())[:3]))

# Reinicia el adaptador para validar observaciones.
marllib_observations = marllib_env.reset(seed=seed)

# Imprime cantidad de observaciones multiagente.
print('observations:', len(marllib_observations))

## Campaign Training

El entrenamiento completo no debe lanzarse accidentalmente desde un QuickStart. La campaña tiene etapas en orden obligatorio: piloto, HPO (semillas de desarrollo 901-903, seleccion en `validation_2024`), congelado de configuracion, confirmacion (4 algoritmos x 3 escenarios x 12 semillas 1001-1012, 50 episodios de 8760 pasos) y sensibilidad (p=0.55/0.65 sobre 1001-1003). Esta celda solo imprime los comandos.

In [ ]:
# Construye los comandos de la CLI de campaña por etapa (solo se imprimen).
for stage in ('pilot', 'hpo', 'confirm', 'sensitivity'):
    stage_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                     '--config', 'configs/experiments/iquitos.yaml', '--stage', stage]
    # Muestra el comando; sin --execute la CLI solo describe la etapa.
    print(' '.join(map(str, stage_command)))

# Para entrenar una etapa autorizada se agrega --execute al comando (ver madrl_citylearn_v3_cli.ipynb).
# Ejemplo, solo tras autorizarla en resource_authorization.authorized_stages:
#   python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage pilot --execute

## Neighborhood Dataset Generation

El QuickStart original muestra como generar barrios residenciales con EnergyPlus. Esa funcionalidad sigue perteneciendo a CityLearn v2. Para este proyecto, cualquier nuevo `schema.json` compatible con CityLearn v2 puede entrar a la capa v3 mediante `make_citylearn_v3_env(schema_path=...)` y conservar Dec-POMDP, CTDE, reward v3 y KPIs por eje.

In [ ]:
# Importa la fabrica generica de CityLearn v3.
from citylearn.v3 import make_citylearn_v3_env

# Usa el schema del split de validacion como ejemplo de entrada generica.
custom_schema_path = DATASET_ROOT / 'splits' / 'validation_2024' / 'schema.json'

# Crea un entorno v3 desde un schema explicito.
custom_env = make_citylearn_v3_env(schema_path=custom_schema_path, scenario='E2', seed=seed, episode_time_steps=quick_steps, madrl_algorithm='MASAC')

# Resume el entorno construido desde schema explicito.
custom_description = describe_environment(custom_env)

# Imprime ruta del schema usado.
print('schema_path:', custom_schema_path)

# Imprime escenario activo.
print('scenario:', custom_description['scenario'])

# Imprime reward activo.
print('reward_function:', custom_description['reward_function'])

# Imprime perfil de reward.
print('reward_profile:', custom_description['reward_metadata']['profile']['profile_name'])

## Quick Evaluation by Project Axes

La evaluacion final del proyecto no se basa solo en reward. Se reportan KPIs CityLearn v2 agrupados por los tres ejes: flexibilidad energetica, emisiones de CO2 y costos energeticos.

In [ ]:
# Evalua objetivos sobre el entorno v3 creado para smoke test.
objective_report = evaluate_objectives(madrl_env)

# Recorre los tres ejes del proyecto.
for axis_name, axis_payload in objective_report['axes'].items():
    # Extrae el resumen de comparacion contra baseline.
    comparison = axis_payload['baseline_comparison']
    # Imprime resumen por eje.
    print(axis_name, comparison)